# fase_3 - script_cimut Migration

This notebook handles migration of database from old DB to new DB for fase 3.

**Purpose**: Benerin database lama ke database baru untuk bagian CRM, Prospek, dan Operasional.

In [25]:
import sys
import os
import mysql.connector
import pandas as pd
import datetime
import random
import string
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## Connect ke Database

In [26]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: 3


## Ambil Data dari DB Lama

In [27]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS
# ---------------------------------------------------------
cursor_old.execute("SHOW TABLES")
tables_data = cursor_old.fetchall()
target_tables = [list(t.values())[0] for t in tables_data]
print(f"\n--- Ditemukan {len(target_tables)} tabel di Database Lama ---")

df_old = {}
for table in target_tables:
    try:
        query = f"SELECT * FROM `{table}`"
        df_old[table] = pd.read_sql(query, db_old)
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_old[table])}")
    except Exception as e:
        print(f"Gagal load tabel {table}: {e}")

print("\n--- Proses load selesai. Semua data tersimpan di 'df_old' ---")


--- Ditemukan 108 tabel di Database Lama ---
Berhasil load tabel: absensi | Jumlah baris: 13444
Berhasil load tabel: absensi_note | Jumlah baris: 11
Berhasil load tabel: bidang | Jumlah baris: 4
Berhasil load tabel: bidangkategori | Jumlah baris: 12
Berhasil load tabel: bidanglink | Jumlah baris: 7
Berhasil load tabel: calon | Jumlah baris: 4
Berhasil load tabel: calon_detil | Jumlah baris: 61
Berhasil load tabel: calon_pertanyaan | Jumlah baris: 229
Berhasil load tabel: calon_pertanyaan_detil | Jumlah baris: 4305
Berhasil load tabel: catatan_kelas | Jumlah baris: 12797
Berhasil load tabel: catatan_kelas_tag | Jumlah baris: 999
Berhasil load tabel: catatan_mingguan | Jumlah baris: 0
Berhasil load tabel: catatan_siswa | Jumlah baris: 1502
Berhasil load tabel: catatan_siswa_follow_up | Jumlah baris: 22
Berhasil load tabel: catatanawal_admin | Jumlah baris: 64
Berhasil load tabel: catatanawal_datautama | Jumlah baris: 9
Berhasil load tabel: catatanawal_infolain | Jumlah baris: 64
Berhasi

## Ambil Data dari DB Baru (Struktur Target)

In [28]:
cursor_new.execute("SHOW TABLES")
tables_data_new = cursor_new.fetchall()
target_tables_new = [list(t.values())[0] for t in tables_data_new]
df_new = {}

for table in target_tables_new:
    try:
        query = f"SELECT * FROM `{table}`"
        df_new[table] = pd.read_sql(query, db_new)
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_new[table])}")
    except:
        pass

Berhasil load tabel: absensi | Jumlah baris: 0
Berhasil load tabel: activity_log | Jumlah baris: 0
Berhasil load tabel: admin_sarpras | Jumlah baris: 1
Berhasil load tabel: bidang_kategori | Jumlah baris: 12
Berhasil load tabel: bidang_link | Jumlah baris: 7
Berhasil load tabel: busdev_bidang | Jumlah baris: 4
Berhasil load tabel: cache | Jumlah baris: 0
Berhasil load tabel: cache_locks | Jumlah baris: 0
Berhasil load tabel: calon_siswa | Jumlah baris: 160
Berhasil load tabel: calon_siswa_akademik | Jumlah baris: 158
Berhasil load tabel: calon_siswa_bayar | Jumlah baris: 166
Berhasil load tabel: calon_siswa_fo_detail | Jumlah baris: 0
Berhasil load tabel: calon_siswa_form_program_requirements | Jumlah baris: 0
Berhasil load tabel: calon_siswa_form_programs | Jumlah baris: 0
Berhasil load tabel: calon_siswa_jadwal | Jumlah baris: 166
Berhasil load tabel: calon_siswa_kursus | Jumlah baris: 166
Berhasil load tabel: calon_siswa_ortu | Jumlah baris: 166
Berhasil load tabel: calon_siswa_pros

In [29]:
display(df_old['format_rapor'])

,idformat_rapor,idpendkursus,title,idpendkursusmitra
0,F00001,K00001,CLASSROOM ASSESSMENT,None
1,F00002,K00001,END OF TERM TEST,None
2,F00003,K00001,CLASS REMARKS,None
3,F00004,K00001,CLASSROOM ASSESSMENT,None
4,F00005,K00001,END OF TERM TEST,None
5,F00006,K00001,CLASS REMARKS,None
6,F00007,K00001,CLASSROOM ASSESSMENT,None
7,F00008,K00001,END OF TERM TEST,None
8,F00009,K00001,CLASS REMARKS,None
9,F00010,K00003,CLASSROOM ASSESSMENT,None


# Target: izin_karyawan, verifikasi_izin, absensi, verifikasi_absensi, karyawan_resign.

In [30]:
print("================================================================================")
# 1. Cek isi tabel perijinan di database lama (df_old)
if 'perijinan' in df_old:
    print("📊 DATA ASLI TABEL 'perijinan' DI DATABASE LAMA:")
    display(df_old['perijinan'].head(10)) # Intip 10 data teratas
    
    print("\n📈 VARIASI STATUS PADA TABEL PERIJINAN LAMA (UNTUK AUDIT ENUM):")
    display(df_old['perijinan']['status'].value_counts(dropna=False))
else:
    print("⚠️ WARNING: Tabel 'perijinan' tidak ditemukan di dalam df_old kamu!")
    print("Daftar tabel yang tersedia di df_old saat ini adalah:", list(df_old.keys()))

print("--------------------------------------------------------------------------------")

# 2. Cek wadah target izin_karyawan di database baru (df_new)
if 'izin_karyawan' in df_new:
    print("📋 STRUKTUR WADAH TARGET 'izin_karyawan' DI DATABASE BARU:")
    df_new['izin_karyawan'].info()
else:
    print("⚠️ WARNING: Wadah df_new['izin_karyawan'] belum terdefinisi di dictionary kamu.")
print("================================================================================")

📊 DATA ASLI TABEL 'perijinan' DI DATABASE LAMA:


,idperijinan,jenis,tanggalmulai,tanggalselesai,waktumulai,waktuselesai,keterangan,surat,idusers,created_at,status,catatan
0,P00003,Ijin,2023-11-10,2023-11-10,0 days 15:30:00,0 days 17:00:00,Al muslin ekskul,None,U00012,2023-11-13 07:34:57,Disetujui,
1,P00004,Lembur,2023-11-13,2023-11-13,0 days 07:00:00,0 days 08:30:00,pengganti Al muslim,None,U00012,2023-11-13 07:35:47,Disetujui,
2,P00007,Lembur,2023-11-26,2023-11-26,0 days 16:00:00,0 days 18:00:00,Mengganti 2 jam kerja Senin 27 November 2023 j...,1701093214_10eabbf62174540924e1.jpeg,U00014,2023-11-27 20:53:34,Disetujui,
3,P00009,Lembur,2023-12-02,2023-12-02,0 days 09:00:00,0 days 13:00:00,"Mengganti 4 jam kerja Kamis, 30 November 2023 ...",1701093670_6ed710a3b721f5b1b08d.pdf,U00014,2023-11-27 21:01:10,Diajukan,None
4,P00010,Ijin,2023-11-29,2023-11-29,0 days 13:00:00,0 days 15:00:00,Les Coding agnes,None,U00003,2023-11-29 13:24:43,Disetujui,
5,P00014,Lembur,2023-12-04,2023-12-04,0 days 13:00:00,0 days 17:00:00,mengganti 2 jam les agnes + 2 jam nabung untuk...,None,U00003,2023-12-04 08:56:46,Disetujui,
6,P00015,Ijin,2023-12-05,2023-12-05,0 days 14:00:00,0 days 15:40:00,Les Coding agnes,None,U00003,2023-12-05 15:29:31,Disetujui,
7,P00017,Lembur,2023-12-11,2023-12-11,0 days 13:00:00,0 days 17:00:00,nabung jam untuk les agnes atau kebutuhan anak...,None,U00003,2023-12-11 14:50:44,Disetujui,
8,P00018,Ijin,2023-12-13,2023-12-13,0 days 13:30:00,0 days 15:00:00,Les Agnes,None,U00003,2023-12-18 11:52:34,Disetujui,
9,P00019,Lembur,2023-12-18,2023-12-18,0 days 13:30:00,0 days 16:30:00,nabung jam untuk anak panah nanti,None,U00003,2023-12-18 11:53:31,Disetujui,



📈 VARIASI STATUS PADA TABEL PERIJINAN LAMA (UNTUK AUDIT ENUM):


status
Disetujui                      902
Diterima oleh Kepala Divisi     35
Diajukan                        19
Ditolak oleh Kepala Divisi       1
Name: count, dtype: int64

--------------------------------------------------------------------------------
📋 STRUKTUR WADAH TARGET 'izin_karyawan' DI DATABASE BARU:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 957 entries, 0 to 956
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype          
---  ------            --------------  -----          
 0   id_izin           957 non-null    int64          
 1   id_karyawan       957 non-null    int64          
 2   jenis_izin        957 non-null    object         
 3   tanggal_mulai     957 non-null    object         
 4   tanggal_selesai   957 non-null    object         
 5   waktu_mulai       957 non-null    timedelta64[ns]
 6   waktu_selesai     957 non-null    timedelta64[ns]
 7   keterangan_izin   957 non-null    object         
 8   dokumen_lampiran  957 non-null    object         
 9   created_at        957 non-null    datetime64[ns] 
dtypes: datetime64[ns](1), int64(2), object(5), timedelta64[ns](2)
memory usage: 74.9+ KB


In [31]:
import pandas as pd
import datetime
import numpy as np

print("================================================================================")
print(" 🚀 FASE 4 - STEP 1: MATCHING SKEMA WADAH BARU (10 COLUMNS FIXED) 🚀 ")
print("================================================================================")

def transform_to_izin_karyawan_final_v6(df_old_source):
    print("🔄 Memulai ekstraksi komponen Tanggal & Jam terpisah sesuai 10 struktur target...")
    
    enum_izin_target_map = {
        'diajukan': 'Pending',
        'disetujui': 'Disetujui',
        'diterima oleh kepala divisi': 'Disetujui',
        'ditolak oleh kepala divisi': 'Ditolak'
    }
    
    izin_records = []
    
    for idx, row in df_old_source.iterrows():
        # 1. Kolom [1]: id_karyawan ditarik dari id_users / idusers database lama
        id_user_raw = str(row.get('id_users') or row.get('idusers') or '').strip()
        id_karyawan_final = id_user_raw if id_user_raw not in ["", "nan", "None", "-"] else None
        
        # 2. Ambil nilai mentah tanggal & waktu dari DB lama
        val_tgl_mulai = row.get('tanggalmulai') or row.get('tgl_mulai')
        val_jam_mulai = row.get('waktumulai') or row.get('jam_mulai')
        
        val_tgl_selesai = row.get('tanggalselesai') or row.get('tgl_selesai')
        val_jam_selesai = row.get('waktuselesai') or row.get('jam_selesai')

        # ---------------------------------------------------------------------
        # 🧼 LOGIKA EKSTRAKSI TANGGAL & WAKTU MULAI (ANTI-00:00:00)
        # ---------------------------------------------------------------------
        # Ekstrak Tanggal Mulai murni (YYYY-MM-DD)
        if pd.notna(val_tgl_mulai):
            tgl_mulai_final = pd.to_datetime(val_tgl_mulai).strftime('%Y-%m-%d')
        else:
            tgl_mulai_final = datetime.date.today().strftime('%Y-%m-%d')
            
        # Ekstrak Waktu Mulai murni (HH:MM:SS)
        if pd.notna(val_jam_mulai):
            if isinstance(val_jam_mulai, (datetime.time, datetime.timedelta)) or 'time' in str(type(val_jam_mulai)).lower():
                waktu_mulai_final = str(val_jam_mulai).split()[-1][:8]
            else:
                waktu_mulai_final = str(val_jam_mulai).strip()[:8]
        else:
            waktu_mulai_final = "08:00:00" # Default masuk jam kerja jika kosong

        # ---------------------------------------------------------------------
        # 🧼 LOGIKA EKSTRAKSI TANGGAL & WAKTU SELESAI (ANTI-00:00:00)
        # ---------------------------------------------------------------------
        # Ekstrak Tanggal Selesai murni (YYYY-MM-DD)
        if pd.notna(val_tgl_selesai):
            tgl_selesai_final = pd.to_datetime(val_tgl_selesai).strftime('%Y-%m-%d')
        else:
            tgl_selesai_final = tgl_mulai_final
            
        # Ekstrak Waktu Selesai murni (HH:MM:SS)
        if pd.notna(val_jam_selesai):
            if isinstance(val_jam_selesai, (datetime.time, datetime.timedelta)) or 'time' in str(type(val_jam_selesai)).lower():
                waktu_selesai_final = str(val_jam_selesai).split()[-1][:8]
            else:
                waktu_selesai_final = str(val_jam_selesai).strip()[:8]
        else:
            waktu_selesai_final = "17:00:00" # Default pulang jam kerja jika kosong

        # ---------------------------------------------------------------------
        # 📝 PEMBERSIHAN KETERANGAN & STATUS
        # ---------------------------------------------------------------------
        alasan_raw = str(row.get('keperluan') or row.get('alasan') or row.get('keterangan') or '').strip()
        keterangan_final = alasan_raw if alasan_raw not in ["", "nan", "None", "-"] else "Tidak ada keterangan"
        
        # ---------------------------------------------------------------------
        # 🗺️ BENTUK BARIS DATA SESUAI 10 KOLOM STRUKTUR BARU KAMU SMURNI
        # ---------------------------------------------------------------------
        record = {
            'id_izin': row.get('idperijinan') or row.get('id_izin') or row.get('idizin'), # Simpan ID asli lama untuk foreign key verifikasi nanti
            'id_karyawan': id_karyawan_final,
            'jenis_izin': row.get('jenis'),
            'tanggal_mulai': tgl_mulai_final,
            'tanggal_selesai': tgl_selesai_final,
            'waktu_mulai': waktu_mulai_final,
            'waktu_selesai': waktu_selesai_final,
            'keterangan_izin': keterangan_final,
            'dokumen_lampiran': row.get('surat') or None,
            'created_at': row.get('created_at') if pd.notna(row.get('created_at')) else datetime.datetime.now()
        }
        izin_records.append(record)
        
    df_result = pd.DataFrame(izin_records)
    print(f"✓ Sukses memetakan data perijinan! Total: {len(df_result)} baris cocok dengan 10 kolom target.")
    return df_result

# === JALANKAN PROSES TRANSFER ===
if 'perijinan' in df_old:
    df_new['izin_karyawan'] = transform_to_izin_karyawan_final_v6(df_old['perijinan'])
    
    print("\n📸 PREVIEW DF_NEW['IZIN_KARYAWAN'] 10 KOLOM FINAL:")
    display(df_new['izin_karyawan'])

 🚀 FASE 4 - STEP 1: MATCHING SKEMA WADAH BARU (10 COLUMNS FIXED) 🚀 
🔄 Memulai ekstraksi komponen Tanggal & Jam terpisah sesuai 10 struktur target...
✓ Sukses memetakan data perijinan! Total: 957 baris cocok dengan 10 kolom target.

📸 PREVIEW DF_NEW['IZIN_KARYAWAN'] 10 KOLOM FINAL:


,id_izin,id_karyawan,jenis_izin,tanggal_mulai,tanggal_selesai,waktu_mulai,waktu_selesai,keterangan_izin,dokumen_lampiran,created_at
0,P00003,U00012,Ijin,2023-11-10,2023-11-10,15:30:00,17:00:00,Al muslin ekskul,None,2023-11-13 07:34:57
1,P00004,U00012,Lembur,2023-11-13,2023-11-13,07:00:00,08:30:00,pengganti Al muslim,None,2023-11-13 07:35:47
2,P00007,U00014,Lembur,2023-11-26,2023-11-26,16:00:00,18:00:00,Mengganti 2 jam kerja Senin 27 November 2023 j...,1701093214_10eabbf62174540924e1.jpeg,2023-11-27 20:53:34
3,P00009,U00014,Lembur,2023-12-02,2023-12-02,09:00:00,13:00:00,"Mengganti 4 jam kerja Kamis, 30 November 2023 ...",1701093670_6ed710a3b721f5b1b08d.pdf,2023-11-27 21:01:10
4,P00010,U00003,Ijin,2023-11-29,2023-11-29,13:00:00,15:00:00,Les Coding agnes,None,2023-11-29 13:24:43
...,...,...,...,...,...,...,...,...,...,...
952,P01002,U00023,Ijin,2026-02-27,2026-02-27,07:00:00,16:05:00,Terlambat,None,2026-04-06 17:01:49
953,P01003,U00023,Ijin,2026-03-31,2026-03-31,07:00:00,17:15:00,Terlambat,None,2026-04-06 17:02:36
954,P01004,U00012,Ijin,2026-04-08,2026-04-08,18:15:00,19:15:00,"ijin pulang lebih cepat karena mau ke bengkel,...",None,2026-04-08 11:40:20
955,P01005,U00033,Lembur,2026-03-31,2026-03-31,09:17:00,10:05:00,"tabungan jam maret, 58 menit",1775649994_ebf653ce91e06066e37c.jpg,2026-04-08 19:06:34


In [32]:
display(df_old['perijinan_note'].info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2107 entries, 0 to 2106
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   idnotes      2107 non-null   int64 
 1   idperijinan  2107 non-null   object
 2   baca         2107 non-null   int64 
 3   status       2107 non-null   object
 4   catatan      1150 non-null   object
 5   idjabatan    174 non-null    object
 6   iddivisi     968 non-null    object
dtypes: int64(2), object(5)
memory usage: 115.4+ KB


None

In [33]:
import pandas as pd
import datetime

print("================================================================================")
print(" 🚀 FASE 4 - STEP 4: MIGRATION MURNI 'perijinan_note' TO 'verifikasi_izin' 🚀 ")
print("================================================================================")

def transform_perijinan_note_to_verifikasi(df_old_note):
    print("🔄 Memindahkan data secara langsung dari tabel perijinan_note...")
    
    if df_old_note is None or df_old_note.empty:
        print("❌ Gagal: Tabel 'perijinan_note' lama kosong!")
        return pd.DataFrame()
        
    records = []
    current_time = datetime.datetime.now()
    
    for idx, row in df_old_note.iterrows():
        # Bersihkan catatan agar jika nilainya NaN tidak merusak insert MySQL
        catatan_raw = row.get('catatan')
        catatan_final = str(catatan_raw).strip() if pd.notna(catatan_raw) else "Tidak ada catatan"
        
        # Susun tepat 7 kolom mengikuti blueprint fisik wadah baru Cimut
        record = {
            'id_verifikasi_izin': row.get('idnotes'),
            'id_izin': row.get('idperijinan'),
            'status_verifikasi_izin': row.get('status'),
            'catatan_verifikator': catatan_final,
            'id_division': row.get('iddivisi') if pd.notna(row.get('iddivisi')) else None,
            'created_at': current_time,
        }
        records.append(record)
        
    df_result = pd.DataFrame(records)
    print(f"✓ Sukses memindahkan data! Total: {len(df_result)} baris masuk ke df_new['verifikasi_izin'].")
    return df_result

# === JALANKAN EKSEKUSI LANGSUNG ===
if 'perijinan_note' in df_old:
    df_new['verifikasi_izin'] = transform_perijinan_note_to_verifikasi(df_old['perijinan_note'])
    
    print("\n📸 PREVIEW DF_NEW['VERIFIKASI_IZIN'] 7 KOLOM FINAL:")
    display(df_new['verifikasi_izin'])

 🚀 FASE 4 - STEP 4: MIGRATION MURNI 'perijinan_note' TO 'verifikasi_izin' 🚀 
🔄 Memindahkan data secara langsung dari tabel perijinan_note...
✓ Sukses memindahkan data! Total: 2107 baris masuk ke df_new['verifikasi_izin'].

📸 PREVIEW DF_NEW['VERIFIKASI_IZIN'] 7 KOLOM FINAL:


,id_verifikasi_izin,id_izin,status_verifikasi_izin,catatan_verifikator,id_division,created_at
0,164,P00003,Diajukan,Tidak ada catatan,D00003,2026-06-09 14:06:37.041912
1,165,P00004,Diajukan,Tidak ada catatan,D00003,2026-06-09 14:06:37.041912
2,167,P00003,Disetujui,,None,2026-06-09 14:06:37.041912
3,168,P00004,Disetujui,,None,2026-06-09 14:06:37.041912
4,173,P00007,Diajukan,Tidak ada catatan,D00003,2026-06-09 14:06:37.041912
...,...,...,...,...,...,...
2102,2384,P01005,Diajukan,Tidak ada catatan,D00003,2026-06-09 14:06:37.041912
2103,2385,P01006,Diajukan,Tidak ada catatan,D00003,2026-06-09 14:06:37.041912
2104,2386,P00998,Diterima oleh Kepala Divisi,,D00003,2026-06-09 14:06:37.041912
2105,2387,P00968,Diterima oleh Kepala Divisi,,D00003,2026-06-09 14:06:37.041912


# absensi

In [34]:
print("================================================================================")
print(" 🔍 INSPEKSI DATA FASE 4: CLUSTER ABSENSI (DB LAMA VS DB BARU) 🚀 ")
print("================================================================================")

# 1. Mengintip Tabel Absensi Lama
if 'absensi' in df_old:
    print("📊 [DB LAMA] TABEL 'absensi':")
    print(f"Total data: {len(df_old['absensi'])} baris")
    display(df_old['absensi'].head(5))
    
    print("\n📈 VARIATION STATUS ABSENSI LAMA (UNTUK STRATEGI ENUM):")
    display(df_old['absensi']['status'].value_counts(dropna=False))
else:
    print("⚠️ WARNING: Tabel 'absensi' tidak ditemukan di df_old!")

print("-" * 80)

# 2. Mengintip Tabel Absensi Note Lama (Jika ada catatan tambahan)
if 'absensi_note' in df_old:
    print("📊 [DB LAMA] TABEL 'absensi_note':")
    print(f"Total data: {len(df_old['absensi_note'])} baris")
    display(df_old['absensi_note'].head(5))
else:
    print("ℹ️ INFO: Tabel 'absensi_note' tidak ditemukan di df_old (Aman, nanti kita gabungkan note1 & note2 dari tabel absensi utama).")

print("-" * 80)

# 3. Mengintip Wadah Target Absensi Baru
if 'absensi' in df_new:
    print("📋 [DB BARU] STRUKTUR WADAH TARGET 'absensi':")
    df_new['absensi'].info()
else:
    print("⚠️ WARNING: Wadah df_new['absensi'] belum terdefinisi di dictionary kamu.")
print("================================================================================")

 🔍 INSPEKSI DATA FASE 4: CLUSTER ABSENSI (DB LAMA VS DB BARU) 🚀 
📊 [DB LAMA] TABEL 'absensi':
Total data: 13444 baris


,idabsensi,tanggal,scanmasuk,scankeluar,status,idkaryawan,created_at,note1,note2,verifikasi,masuk,terlambat,keluar,cepat
0,309,2023-06-08,0 days 20:32:35,NaT,Tepat Waktu,LEAP026VIII2021,2023-06-09 08:32:35,<p>makan dulu</p>,None,,NaT,NaT,NaT,NaT
1,310,2023-06-11,0 days 23:40:44,NaT,Tepat Waktu,LEAP026VIII2021,2023-06-12 11:40:44,<p>test 2</p>,None,,NaT,NaT,NaT,NaT
2,311,2023-06-12,0 days 00:02:52,0 days 00:03:06,Tepat Waktu,LEAP011XII01,2023-06-12 12:02:52,<p>fu muh</p>,<p>fu muh</p>,,NaT,NaT,NaT,NaT
3,312,2023-06-12,0 days 04:52:25,0 days 00:00:00,Tepat Waktu,LEAP012III02,2023-06-12 16:52:25,<p>Mau tiduran</p>,None,,NaT,NaT,NaT,NaT
4,313,2023-06-01,0 days 00:00:00,0 days 00:00:00,Tidak Hadir,LEAP019VI2019,2023-06-28 15:03:05,None,None,None,NaT,NaT,NaT,NaT



📈 VARIATION STATUS ABSENSI LAMA (UNTUK STRATEGI ENUM):


status
Tepat Waktu    8450
Alpha          1804
Terlambat      1234
Hadir          1006
Tidak Hadir     937
Libur             9
Sakit             3
                  1
Name: count, dtype: int64

--------------------------------------------------------------------------------
📊 [DB LAMA] TABEL 'absensi_note':
Total data: 11 baris


,idnote,catatan,created_at
0,1,<p>dcvbhnjk</p>,2023-05-17 15:54:46
1,3,<p>sudah oke untuk absensi</p>,2023-07-18 18:21:38
2,4,<p>okee</p>,2024-11-30 00:00:00
3,5,,2024-10-01 00:00:00
4,6,<p>Sudah ACC</p>,2024-04-01 00:00:00


--------------------------------------------------------------------------------
📋 [DB BARU] STRUKTUR WADAH TARGET 'absensi':
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_absensi             0 non-null      object
 1   id_karyawan            0 non-null      object
 2   id_izin                0 non-null      object
 3   tanggal                0 non-null      object
 4   jam_masuk              0 non-null      object
 5   jam_keluar             0 non-null      object
 6   catatan_masuk          0 non-null      object
 7   catatan_keluar         0 non-null      object
 8   status_absensi         0 non-null      object
 9   tipe_absensi           0 non-null      object
 10  id_verifikasi_absensi  0 non-null      object
 11  created_at             0 non-null      object
dtypes: object(12)
memory usage: 132.0+ bytes


In [35]:
import pandas as pd
import datetime
import numpy as np

print("================================================================================")
print(" 🚀 FASE 4 - STEP 2: TAHAP 1 - PERPINDAHAN ABSENSI MENTAH (RAW MIGRATION) 🚀 ")
print("================================================================================")

def transform_absensi_raw(df_old_absensi):
    print("🔄 Mentransfer seluruh log harian absensi lama ke skema wadah baru...")
    
    enum_absen_target_map = {
        'tepat waktu': 'Tepat Waktu',
        'hadir': 'Hadir',
        'terlambat': 'Terlambat',
        'sakit': 'Izin',
        'tidak hadir': 'Izin',
        'alpha': 'Izin',
        'libur': 'Hadir'
    }
    
    absensi_records = []
    
    for idx, row in df_old_absensi.iterrows():
        # A. Ambil ID Karyawan murni teks
        id_kar_raw = str(row.get('idkaryawan') or row.get('id_karyawan') or row.get('idusers') or '').strip()
        id_karyawan_final = id_kar_raw if id_kar_raw not in ["", "nan", "None", "-"] else None
        
        # B. Ambil Tanggal Absen Bersih
        raw_tanggal = str(row.get('tanggal') or '').strip()
        tanggal_clean = raw_tanggal[:10] if len(raw_tanggal) >= 10 else None
        
        if not tanggal_clean:
            continue
            
        # C. Penyesuaian Status ENUM Target
        status_raw = str(row.get('status') or '').strip().lower()
        status_absensi_final = enum_absen_target_map.get(status_raw, 'Hadir')
        
        # D. Ambil komponen Jam Masuk & Keluar Bersih
        jam_masuk_raw = str(row.get('scanmasuk') or row.get('jam_masuk') or '').strip()
        jam_masuk_final = jam_masuk_raw.split()[-1][:8] if jam_masuk_raw not in ["", "nan", "None", "-"] else None
        
        jam_keluar_raw = str(row.get('scankeluar') or row.get('jam_keluar') or '').strip()
        jam_keluar_final = jam_keluar_raw.split()[-1][:8] if jam_keluar_raw not in ["", "nan", "None", "-"] else None

        # E. Ambil Catatan Masuk & Keluar
        catat_masuk_final = row.get('note1') if pd.notna(row.get('note1')) else None
        catat_keluar_final = row.get('note2') if pd.notna(row.get('note2')) else None

        # Susun baris baru mengikuti 12 kolom target baru (id_izin sementara kita kosongkan dulu)
        record = {
            'id_absensi': row.get('idabsensi') or row.get('id_absensi'),
            'id_karyawan': id_karyawan_final,
            'id_izin': None, # 💡 Ditahap ini sengaja kita kosongkan murni None (NULL)
            'tanggal': tanggal_clean,
            'jam_masuk': jam_masuk_final,
            'jam_keluar': jam_keluar_final,
            'catatan_masuk': catat_masuk_final,
            'catatan_keluar': catat_keluar_final,
            'status_absensi': status_absensi_final,
            'tipe_absensi': 'Fingerprint',
            'id_verifikasi_absensi': None,
            'created_at': row.get('created_at') if pd.notna(row.get('created_at')) else datetime.datetime.now()
        }
        absensi_records.append(record)
        
    df_result = pd.DataFrame(absensi_records)
    print(f"✓ Sukses memindahkan {len(df_result)} baris absensi mentah ke df_new['absensi'].")
    return df_result

# Eksekusi Tahap 1
if 'absensi' in df_old:
    df_new['absensi'] = transform_absensi_raw(df_old['absensi'])
    display(df_new['absensi'].head(5))

 🚀 FASE 4 - STEP 2: TAHAP 1 - PERPINDAHAN ABSENSI MENTAH (RAW MIGRATION) 🚀 
🔄 Mentransfer seluruh log harian absensi lama ke skema wadah baru...
✓ Sukses memindahkan 13444 baris absensi mentah ke df_new['absensi'].


,id_absensi,id_karyawan,id_izin,tanggal,jam_masuk,jam_keluar,catatan_masuk,catatan_keluar,status_absensi,tipe_absensi,id_verifikasi_absensi,created_at
0,309,LEAP026VIII2021,None,2023-06-08,20:32:35,NaT,<p>makan dulu</p>,None,Tepat Waktu,Fingerprint,None,2023-06-09 08:32:35
1,310,LEAP026VIII2021,None,2023-06-11,23:40:44,NaT,<p>test 2</p>,None,Tepat Waktu,Fingerprint,None,2023-06-12 11:40:44
2,311,LEAP011XII01,None,2023-06-12,00:02:52,00:03:06,<p>fu muh</p>,<p>fu muh</p>,Tepat Waktu,Fingerprint,None,2023-06-12 12:02:52
3,312,LEAP012III02,None,2023-06-12,04:52:25,None,<p>Mau tiduran</p>,None,Tepat Waktu,Fingerprint,None,2023-06-12 16:52:25
4,313,LEAP019VI2019,None,2023-06-01,None,None,None,None,Izin,Fingerprint,None,2023-06-28 15:03:05


In [36]:
print("================================================================================")
print(" 🕵️‍♂️ FASE 4 - STEP 2: TAHAP 2 - DATA LINKING SUNTIK ID_IZIN PANDAS INDEPENDEN 🕵️‍♂️")
print("================================================================================")

def inject_id_izin_into_absensi(df_target_absensi, df_reference_izin):
    if df_target_absensi is None or df_target_absensi.empty:
        print("❌ Gagal: Data target absensi kosong!")
        return df_target_absensi
        
    if df_reference_izin is None or df_reference_izin.empty:
        print("⚠️ Peringatan: Data acuan izin kosong, proses penyuntikan dilewati.")
        return df_target_absensi

    print("🔄 Mempersiapkan data acuan izin dari df_new['izin_karyawan']...")
    
    # Kumpulkan acuan izin ke dalam list pencarian murni tanggal (normalize)
    list_acuan_izin = []
    for _, iz_row in df_reference_izin.iterrows():
        if pd.notna(iz_row.get('id_karyawan')) and pd.notna(iz_row.get('tanggal_mulai')):
            list_acuan_izin.append({
                'id_izin': iz_row.get('id_izin'),
                'id_karyawan': str(iz_row.get('id_karyawan')).strip().lower(),
                'mulai_dt': pd.to_datetime(iz_row.get('tanggal_mulai')).normalize(),
                'selesai_dt': pd.to_datetime(iz_row.get('tanggal_selesai')).normalize()
            })

    print("🔎 Melakukan pemindaian interval tanggal murni pada baris berstatus 'Izin'...")
    matched_count = 0
    
    # Kita sisir baris demi baris di df_new['absensi'] yang barusan kita buat di Tahap 1
    for idx, row in df_target_absensi.iterrows():
        # Jalankan logika pencocokan HANYA jika status absensinya diarahkan ke 'Izin'
        if row['status_absensi'] == 'Izin' and pd.notna(row['id_karyawan']) and pd.notna(row['tanggal']):
            try:
                tgl_absen_dt = pd.to_datetime(row['tanggal']).normalize()
                id_kar_lower = str(row['id_karyawan']).strip().lower()
                
                # Adu dengan list acuan perizinan bersih kita
                for iz in list_acuan_izin:
                    if iz['id_karyawan'] == id_kar_lower:
                        if iz['mulai_dt'] <= tgl_absen_dt <= iz['selesai_dt']:
                            # 🔥 SUNTIKKAN nilai id_izin langsung ke indeks baris dataframe ini!
                            df_target_absensi.at[idx, 'id_izin'] = iz['id_izin']
                            matched_count += 1
                            break
            except Exception as e:
                continue

    print(f"================================================================================")
    print(f"🎉 SELESAI! Berhasil menyuntikkan {matched_count} data id_izin ke dalam log absensi.")
    print(f"================================================================================")
    return df_target_absensi

# Jalankan Suntikan Tahap 2
df_new['absensi'] = inject_id_izin_into_absensi(df_new['absensi'], df_new.get('izin_karyawan'))

# Tampilkan hasil setelah penyuntikan untuk pembuktian audit data
print("\n📸 PREVIEW HASIL AKHIR SETELAH DATA LINKING (Cek Kolom id_izin):")
display(df_new['absensi'])

 🕵️‍♂️ FASE 4 - STEP 2: TAHAP 2 - DATA LINKING SUNTIK ID_IZIN PANDAS INDEPENDEN 🕵️‍♂️
🔄 Mempersiapkan data acuan izin dari df_new['izin_karyawan']...
🔎 Melakukan pemindaian interval tanggal murni pada baris berstatus 'Izin'...
🎉 SELESAI! Berhasil menyuntikkan 0 data id_izin ke dalam log absensi.

📸 PREVIEW HASIL AKHIR SETELAH DATA LINKING (Cek Kolom id_izin):


,id_absensi,id_karyawan,id_izin,tanggal,jam_masuk,jam_keluar,catatan_masuk,catatan_keluar,status_absensi,tipe_absensi,id_verifikasi_absensi,created_at
0,309,LEAP026VIII2021,None,2023-06-08,20:32:35,NaT,<p>makan dulu</p>,None,Tepat Waktu,Fingerprint,None,2023-06-09 08:32:35
1,310,LEAP026VIII2021,None,2023-06-11,23:40:44,NaT,<p>test 2</p>,None,Tepat Waktu,Fingerprint,None,2023-06-12 11:40:44
2,311,LEAP011XII01,None,2023-06-12,00:02:52,00:03:06,<p>fu muh</p>,<p>fu muh</p>,Tepat Waktu,Fingerprint,None,2023-06-12 12:02:52
3,312,LEAP012III02,None,2023-06-12,04:52:25,None,<p>Mau tiduran</p>,None,Tepat Waktu,Fingerprint,None,2023-06-12 16:52:25
4,313,LEAP019VI2019,None,2023-06-01,None,None,None,None,Izin,Fingerprint,None,2023-06-28 15:03:05
...,...,...,...,...,...,...,...,...,...,...,...,...
13439,14121,LEAP054VII2024,None,2026-04-17,None,None,None,None,Izin,Fingerprint,None,2026-04-20 09:31:23
13440,14122,LEAP055VIII2024,None,2026-04-17,None,None,None,None,Izin,Fingerprint,None,2026-04-20 09:31:23
13441,14123,LEAP060VIII2025,None,2026-04-17,08:47:00,17:04:00,None,None,Tepat Waktu,Fingerprint,None,2026-04-20 09:31:23
13442,14124,LEAP062X2025,None,2026-04-17,07:31:00,17:02:00,None,None,Tepat Waktu,Fingerprint,None,2026-04-20 09:31:23


In [37]:
# print("================================================================================")
# print(" 🕵️‍♂️ FASE 4 - STEP 2: TAHAP 2 - DATA LINKING MURNI RENTANG TANGGAL (NO ID_KARYAWAN FILTER) 🕵️‍♂️")
# print("================================================================================")

# def inject_id_izin_by_date_only(df_target_absensi, df_reference_izin):
#     if df_target_absensi is None or df_target_absensi.empty:
#         print("❌ Gagal: Data target absensi kosong!")
#         return df_target_absensi
        
#     if df_reference_izin is None or df_reference_izin.empty:
#         print("⚠️ Peringatan: Data acuan izin kosong, proses dilewati.")
#         return df_target_absensi

#     print("🔄 Mempersiapkan data acuan seluruh rentang tanggal izin dari df_new['izin_karyawan']...")
    
#     # Kumpulkan semua izin murni berdasarkan rentang tanggal objek Pandas (normalize)
#     list_acuan_izin = []
#     for _, iz_row in df_reference_izin.iterrows():
#         if pd.notna(iz_row.get('tanggal_mulai')):
#             list_acuan_izin.append({
#                 'id_izin': iz_row.get('id_izin'),
#                 'mulai_dt': pd.to_datetime(iz_row.get('tanggal_mulai')).normalize(),
#                 'selesai_dt': pd.to_datetime(iz_row.get('tanggal_selesai')).normalize()
#             })

#     print("🔎 Melakukan pemindaian tanggal murni pada baris absen berstatus 'Izin'...")
#     matched_count = 0
    
#     # Kita sisir baris demi baris di df_new['absensi']
#     for idx, row in df_target_absensi.iterrows():
#         # Filter tetap berjalan HANYA untuk baris yang status absensinya 'Izin'
#         if row['status_absensi'] == 'Izin' and pd.notna(row['text_tanggal'] if 'text_tanggal' in row else row['tanggal']):
#             try:
#                 # Amankan tanggal absen menjadi objek datetime murni
#                 tgl_absen_dt = pd.to_datetime(row['tanggal']).normalize()
                
#                 # 🔥 LOGIKA BARU CIMUT: Langsung adu ke rentang tanggal tanpa peduli ID Karyawan!
#                 for iz in list_acuan_izin:
#                     if iz['mulai_dt'] <= tgl_absen_dt <= iz['selesai_dt']:
#                         # Suntikkan id_izin yang harinya cocok
#                         df_target_absensi.at[idx, 'id_izin'] = iz['id_izin']
#                         matched_count += 1
#                         break # Berhenti mencari jika tanggalnya sudah tersangkut di salah satu izin
#             except Exception as e:
#                 continue

#     print(f"================================================================================")
#     print(f"🎉 SELESAI! Taktik Tanggal Murni Berhasil Menyuntikkan {matched_count} data id_izin.")
#     print(f"================================================================================")
#     return df_target_absensi

# # 🔥 JALANKAN PROSES SUNTIK TANPA FILTER ID KARYAWAN 🔥
# df_new['absensi'] = inject_id_izin_by_date_only(df_new['absensi'], df_new.get('izin_karyawan'))

# # Tampilkan hasil preview kolom id_izin
# print("\n📸 PREVIEW HASIL AKHIR DATA LINKING TANGGAL MURNI:")
# display(df_new['absensi'])

# # Simpan Backup Aman
# df_new['absensi'].to_
# ('df_new_absensi.pkl')

In [38]:
df_old['absensi_note'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   idnote      11 non-null     int64         
 1   catatan     11 non-null     object        
 2   created_at  11 non-null     datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 396.0+ bytes


In [39]:
import pandas as pd
import datetime

print("================================================================================")
print(" 🚀 FASE 4 - STEP 3: MIGRATION MURNI 'absensi_note' TO 'verifikasi_absensi' 🚀 ")
print("================================================================================")

def transform_absensi_note_to_verifikasi(df_old_note):
    print("🔄 Memindahkan data secara langsung dari tabel absensi_note...")
    
    if df_old_note is None or df_old_note.empty:
        print("❌ Gagal: Tabel 'absensi_note' lama kosong!")
        return pd.DataFrame(columns=['id_verifikasi_absensi', 'status_verifikasi_absensi', 'catatan_atasan', 'created_at'])
        
    records = []
    
    for idx, row in df_old_note.iterrows():
        # Susun 4 kolom pas sesuai cetakan wadah baru dataleap kamu
        record = {
            'id_verifikasi_absensi': row.get('idnote'),
            'status_verifikasi_absensi': "Disetujui",          # Ga usah diisi (NULL)
            'catatan_atasan': row.get('catatan'),        # catatan_atasan = catatan
            'created_at': row.get('created_at')          # created_at = created_at
        }
        records.append(record)
        
    df_result = pd.DataFrame(records)
    print(f"✓ Sukses memindahkan data! Total: {len(df_result)} baris masuk ke df_new['verifikasi_absensi'].")
    return df_result

# === EKSEKUSI LANGSUNG ===
if 'absensi_note' in df_old:
    df_new['verifikasi_absensi'] = transform_absensi_note_to_verifikasi(df_old['absensi_note'])
    
    print("\n📸 PREVIEW DF_NEW['VERIFIKASI_ABSENSI'] 4 KOLOM FINAL (STRIP DOWN VERSION):")
    display(df_new['verifikasi_absensi'])

 🚀 FASE 4 - STEP 3: MIGRATION MURNI 'absensi_note' TO 'verifikasi_absensi' 🚀 
🔄 Memindahkan data secara langsung dari tabel absensi_note...
✓ Sukses memindahkan data! Total: 11 baris masuk ke df_new['verifikasi_absensi'].

📸 PREVIEW DF_NEW['VERIFIKASI_ABSENSI'] 4 KOLOM FINAL (STRIP DOWN VERSION):


,id_verifikasi_absensi,status_verifikasi_absensi,catatan_atasan,created_at
0,1,Disetujui,<p>dcvbhnjk</p>,2023-05-17 15:54:46
1,3,Disetujui,<p>sudah oke untuk absensi</p>,2023-07-18 18:21:38
2,4,Disetujui,<p>okee</p>,2024-11-30 00:00:00
3,5,Disetujui,,2024-10-01 00:00:00
4,6,Disetujui,<p>Sudah ACC</p>,2024-04-01 00:00:00
5,7,Disetujui,<p>Sudah ACC</p>,2024-06-01 00:00:00
6,8,Disetujui,<p>Sudah ACC</p>,2024-08-01 00:00:00
7,9,Disetujui,<p>Sudah ACC</p>,2024-08-01 00:00:00
8,10,Disetujui,<p>Sudah ACC</p>,2024-09-01 00:00:00
9,11,Disetujui,<p>Sudah ACC</p>,2024-05-01 00:00:00


In [40]:
# Sinkronisasi menyuntikkan ID Verifikasi balik ke dalam tabel absensi utama
if 'absensi' in df_new and 'verifikasi_absensi' in df_new:
    df_new['absensi']['id_verifikasi_absensi'] = df_new['absensi']['id_absensi']
    print("✓ Kolom id_verifikasi_absensi di tabel absensi berhasil disinkronkan!")

✓ Kolom id_verifikasi_absensi di tabel absensi berhasil disinkronkan!


In [41]:
df_new['absensi']

,id_absensi,id_karyawan,id_izin,tanggal,jam_masuk,jam_keluar,catatan_masuk,catatan_keluar,status_absensi,tipe_absensi,id_verifikasi_absensi,created_at
0,309,LEAP026VIII2021,None,2023-06-08,20:32:35,NaT,<p>makan dulu</p>,None,Tepat Waktu,Fingerprint,309,2023-06-09 08:32:35
1,310,LEAP026VIII2021,None,2023-06-11,23:40:44,NaT,<p>test 2</p>,None,Tepat Waktu,Fingerprint,310,2023-06-12 11:40:44
2,311,LEAP011XII01,None,2023-06-12,00:02:52,00:03:06,<p>fu muh</p>,<p>fu muh</p>,Tepat Waktu,Fingerprint,311,2023-06-12 12:02:52
3,312,LEAP012III02,None,2023-06-12,04:52:25,None,<p>Mau tiduran</p>,None,Tepat Waktu,Fingerprint,312,2023-06-12 16:52:25
4,313,LEAP019VI2019,None,2023-06-01,None,None,None,None,Izin,Fingerprint,313,2023-06-28 15:03:05
...,...,...,...,...,...,...,...,...,...,...,...,...
13439,14121,LEAP054VII2024,None,2026-04-17,None,None,None,None,Izin,Fingerprint,14121,2026-04-20 09:31:23
13440,14122,LEAP055VIII2024,None,2026-04-17,None,None,None,None,Izin,Fingerprint,14122,2026-04-20 09:31:23
13441,14123,LEAP060VIII2025,None,2026-04-17,08:47:00,17:04:00,None,None,Tepat Waktu,Fingerprint,14123,2026-04-20 09:31:23
13442,14124,LEAP062X2025,None,2026-04-17,07:31:00,17:02:00,None,None,Tepat Waktu,Fingerprint,14124,2026-04-20 09:31:23


# Resign

In [42]:
print("================================================================================")
print(" 🔍 INSPEKSI DATA: TABEL LAMA 'keluar' VS WADAH BARU 'karyawan_resign' 🚀 ")
print("================================================================================")

# 1. Cek struktur data dan isi tabel keluar di database lama
if 'keluar' in df_old:
    print("📊 [DB LAMA] STRUKTUR & CONTOH DATA TABEL 'keluar':")
    print(f"Total data: {len(df_old['keluar'])} baris\n")
    
    # Menampilkan info kolom beserta tipe datanya
    df_old['keluar'].info()
    
    print("\n📸 CONTOH ISI DATA TABEL 'keluar':")
    display(df_old['keluar'])
else:
    print("⚠️ WARNING: Tabel 'keluar' tidak ditemukan di df_old kamu!")
    print("Daftar tabel yang ada di df_old saat ini adalah:", list(df_old.keys()))

print("-" * 80)

# 2. Cek wadah target karyawan_resign di database baru
if 'karyawan_resign' in df_new:
    print("📋 [DB BARU] STRUKTUR WADAH TARGET 'karyawan_resign':")
    df_new['karyawan_resign'].info()
else:
    print("⚠️ WARNING: Wadah df_new['karyawan_resign'] belum terdefinisi.")
print("================================================================================")

 🔍 INSPEKSI DATA: TABEL LAMA 'keluar' VS WADAH BARU 'karyawan_resign' 🚀 
📊 [DB LAMA] STRUKTUR & CONTOH DATA TABEL 'keluar':
Total data: 51 baris

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   idkeluar    51 non-null     int64         
 1   idusers     51 non-null     object        
 2   setuju      49 non-null     float64       
 3   kirim       3 non-null      float64       
 4   alasan      4 non-null      object        
 5   scan        3 non-null      object        
 6   created_at  51 non-null     datetime64[ns]
 7   catatan     1 non-null      object        
 8   status      3 non-null      object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 3.7+ KB

📸 CONTOH ISI DATA TABEL 'keluar':


,idkeluar,idusers,setuju,kirim,alasan,scan,created_at,catatan,status
0,3,U00003,1.0,1.0,menikah dan fokus rumah tangga,1760930076_1c9467f497cf1dfb4157.pdf,2023-04-05 16:18:11,None,Diajukan
1,4,U00001,1.0,NaN,None,None,2023-04-05 16:18:11,None,None
2,11,U00011,1.0,1.0,ikut suami,1764844234_16b2d6c6fe2b556d77c4.pdf,2023-05-25 09:20:40,None,Diajukan
3,12,U00012,NaN,NaN,None,None,2023-05-29 13:48:56,None,None
4,14,U00014,0.0,NaN,None,None,2023-05-29 13:59:36,None,None
5,15,U00015,0.0,NaN,None,None,2023-05-29 14:06:46,None,None
6,18,U00016,0.0,NaN,None,None,2023-05-29 14:21:56,None,None
7,20,U00018,NaN,NaN,None,None,2023-05-30 06:11:17,None,None
8,21,U00019,0.0,NaN,None,None,2023-05-30 15:30:25,None,None
9,22,U00020,0.0,NaN,None,None,2023-05-30 15:32:59,None,None


--------------------------------------------------------------------------------
📋 [DB BARU] STRUKTUR WADAH TARGET 'karyawan_resign':
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_resign           51 non-null     int64         
 1   id_karyawan         51 non-null     int64         
 2   id_user             51 non-null     object        
 3   alasan_resign       51 non-null     object        
 4   dokumen_pendukung   51 non-null     object        
 5   status_persetujuan  51 non-null     object        
 6   status_pengiriman   51 non-null     object        
 7   created_at          51 non-null     datetime64[ns]
dtypes: datetime64[ns](1), int64(2), object(5)
memory usage: 3.3+ KB


In [43]:
import pandas as pd
import numpy as np

print("================================================================================")
print(" 🚀 FASE 4 - STEP 5: MIGRATION MURNI 'keluar' TO 'karyawan_resign' (FIX KEY) 🚀 ")
print("================================================================================")

def transform_keluar_to_karyawan_resign_final(df_old_keluar):
    print("🔄 Memindahkan data secara langsung dari tabel df_old['keluar']...")
    
    if df_old_keluar is None or df_old_keluar.empty:
        print("❌ Gagal: Tabel 'keluar' di df_old kosong atau tidak ditemukan!")
        return pd.DataFrame(columns=[
            'id_resign', 'id_karyawan', 'id_user', 'alasan_resign', 
            'dokumen_pendukung', 'status_persetujuan', 'status_pengiriman', 'created_at'
        ])
        
    records = []
    
    for idx, row in df_old_keluar.iterrows():
        # Pengkondisian alasan resign: utamakan kolom 'alasan', jika kosong intip 'catatan'
        alasan_raw = row.get('alasan')
        catatan_raw = row.get('catatan')
        if pd.notna(alasan_raw) and str(alasan_raw).strip() not in ["", "nan", "None"]:
            alasan_final = str(alasan_raw).strip()
        elif pd.notna(catatan_raw) and str(catatan_raw).strip() not in ["", "nan", "None"]:
            alasan_final = str(catatan_raw).strip()
        else:
            alasan_final = "Tidak ada keterangan"

        # Susun tepat 8 kolom mengikuti cetakan cetak fisik wadah baru Cimut
        record = {
            'id_resign': row.get('idkeluar'),
            'id_karyawan': row.get('idusers'),
            'id_user': row.get('idusers'),
            'alasan_resign': alasan_final,
            'dokumen_pendukung': row.get('scan') if pd.notna(row.get('scan')) else None,
            'status_persetujuan': row.get('status'),
            'status_pengiriman': row.get('kirim') if pd.notna(row.get('kirim')) else None,
            'created_at': row.get('created_at') if pd.notna(row.get('created_at')) else None
        }
        records.append(record)
        
    df_result = pd.DataFrame(records)
    print(f"✓ Sukses memindahkan data! Total: {len(df_result)} baris masuk ke df_new['karyawan_resign'].")
    return df_result

# === JALANKAN EKSEKUSI LANGSUNG MENGGUNAKAN KEY 'keluar' ===
if 'keluar' in df_old:
    df_new['karyawan_resign'] = transform_keluar_to_karyawan_resign_final(df_old['keluar'])
    
    print("\n📸 PREVIEW DF_NEW['KARYAWAN_RESIGN'] 8 KOLOM FINAL:")
    display(df_new['karyawan_resign'])

else:
    print("❌ ERROR: Key 'keluar' beneran gak ada di df_old kamu, Mut. Coba check lagi ngetik namanya.")

 🚀 FASE 4 - STEP 5: MIGRATION MURNI 'keluar' TO 'karyawan_resign' (FIX KEY) 🚀 
🔄 Memindahkan data secara langsung dari tabel df_old['keluar']...
✓ Sukses memindahkan data! Total: 51 baris masuk ke df_new['karyawan_resign'].

📸 PREVIEW DF_NEW['KARYAWAN_RESIGN'] 8 KOLOM FINAL:


,id_resign,id_karyawan,id_user,alasan_resign,dokumen_pendukung,status_persetujuan,status_pengiriman,created_at
0,3,U00003,U00003,menikah dan fokus rumah tangga,1760930076_1c9467f497cf1dfb4157.pdf,Diajukan,1.0,2023-04-05 16:18:11
1,4,U00001,U00001,Tidak ada keterangan,None,None,NaN,2023-04-05 16:18:11
2,11,U00011,U00011,ikut suami,1764844234_16b2d6c6fe2b556d77c4.pdf,Diajukan,1.0,2023-05-25 09:20:40
3,12,U00012,U00012,Tidak ada keterangan,None,None,NaN,2023-05-29 13:48:56
4,14,U00014,U00014,Tidak ada keterangan,None,None,NaN,2023-05-29 13:59:36
5,15,U00015,U00015,Tidak ada keterangan,None,None,NaN,2023-05-29 14:06:46
6,18,U00016,U00016,Tidak ada keterangan,None,None,NaN,2023-05-29 14:21:56
7,20,U00018,U00018,Tidak ada keterangan,None,None,NaN,2023-05-30 06:11:17
8,21,U00019,U00019,Tidak ada keterangan,None,None,NaN,2023-05-30 15:30:25
9,22,U00020,U00020,Tidak ada keterangan,None,None,NaN,2023-05-30 15:32:59


# Update id_karyawan

In [44]:
df_new['absensi']

,id_absensi,id_karyawan,id_izin,tanggal,jam_masuk,jam_keluar,catatan_masuk,catatan_keluar,status_absensi,tipe_absensi,id_verifikasi_absensi,created_at
0,309,LEAP026VIII2021,None,2023-06-08,20:32:35,NaT,<p>makan dulu</p>,None,Tepat Waktu,Fingerprint,309,2023-06-09 08:32:35
1,310,LEAP026VIII2021,None,2023-06-11,23:40:44,NaT,<p>test 2</p>,None,Tepat Waktu,Fingerprint,310,2023-06-12 11:40:44
2,311,LEAP011XII01,None,2023-06-12,00:02:52,00:03:06,<p>fu muh</p>,<p>fu muh</p>,Tepat Waktu,Fingerprint,311,2023-06-12 12:02:52
3,312,LEAP012III02,None,2023-06-12,04:52:25,None,<p>Mau tiduran</p>,None,Tepat Waktu,Fingerprint,312,2023-06-12 16:52:25
4,313,LEAP019VI2019,None,2023-06-01,None,None,None,None,Izin,Fingerprint,313,2023-06-28 15:03:05
...,...,...,...,...,...,...,...,...,...,...,...,...
13439,14121,LEAP054VII2024,None,2026-04-17,None,None,None,None,Izin,Fingerprint,14121,2026-04-20 09:31:23
13440,14122,LEAP055VIII2024,None,2026-04-17,None,None,None,None,Izin,Fingerprint,14122,2026-04-20 09:31:23
13441,14123,LEAP060VIII2025,None,2026-04-17,08:47:00,17:04:00,None,None,Tepat Waktu,Fingerprint,14123,2026-04-20 09:31:23
13442,14124,LEAP062X2025,None,2026-04-17,07:31:00,17:02:00,None,None,Tepat Waktu,Fingerprint,14124,2026-04-20 09:31:23


In [45]:
import pandas as pd
import pickle

print("================================================================================")
print(" 🔄 FASE 4 - UPDATE ID KARYAWAN (VARCHAR -> INT AUTO INCREMENT) 🔄 ")
print("================================================================================")

# 1. LOAD DATA MAPPING DARI FASE 2
# Sesuaikan path-nya jika folder fase_2 ada di direktori yang berbeda
path_mapping = '../fase_2/mapping_id_karyawan.pkl' 

try:
    with open(path_mapping, 'rb') as f:
        df_mapping = pickle.load(f)
    print("✓ Berhasil memuat file mapping_id_karyawan.pkl")
except FileNotFoundError:
    print(f"❌ File tidak ditemukan di path: {path_mapping}. Tolong sesuaikan path-nya ya, Mut!")

# 2. BUAT KAMUS (DICTIONARY) MAPPING
# Key   : id_karyawan_lama (Varchar lama, misal 'U00012' atau 'LEAP...')
# Value : id_karyawan (Integer auto increment baru)
# --- BAGIAN INI YANG DIUPDATE Sesuai nama kolom barumu ---
dict_mapping_karyawan = dict(zip(df_mapping['id_karyawan_lama'], df_mapping['id_karyawan']))

# 3. EKSEKUSI UPDATE KE TABEL-TABEL TERKAIT

# C. UPDATE ABSENSI (Bonus: Karena di kodemu tabel ini juga masih pakai varchar lama)
if 'absensi' in df_new:
    print("\nMemperbarui id_karyawan di tabel absensi...")
    df_new['absensi']['id_karyawan'] = df_new['absensi']['id_karyawan'].map(dict_mapping_karyawan)
    
    # Intip hasil
    display(df_new['absensi'][['id_absensi', 'id_karyawan', 'status_absensi']])

print("\n✅ UPDATE SELESAI! Kolom id_karyawan sekarang sudah berupa Integer.")

 🔄 FASE 4 - UPDATE ID KARYAWAN (VARCHAR -> INT AUTO INCREMENT) 🔄 
✓ Berhasil memuat file mapping_id_karyawan.pkl

Memperbarui id_karyawan di tabel absensi...


,id_absensi,id_karyawan,status_absensi
0,309,12,Tepat Waktu
1,310,12,Tepat Waktu
2,311,3,Tepat Waktu
3,312,4,Tepat Waktu
4,313,9,Izin
...,...,...,...
13439,14121,36,Izin
13440,14122,37,Izin
13441,14123,42,Tepat Waktu
13442,14124,44,Tepat Waktu



✅ UPDATE SELESAI! Kolom id_karyawan sekarang sudah berupa Integer.


In [46]:
df_mapping

,id_karyawan,kode_karyawan,id_karyawan_lama,id_user
0,1,LEAP00102VI23,LEAP001VI02,U00001
1,2,LEAP00313III23,LEAP003III2023,U00003
2,3,LEAP01101XII20,LEAP011XII01,U00011
3,4,LEAP01202III20,LEAP012III02,U00012
4,5,LEAP01431VII18,LEAP014VII31,U00014
5,6,LEAP01514II11,LEAP015II2011,U00015
6,7,LEAP01619VI17,LEAP016VI2017,U00016
7,8,LEAP01820IV21,LEAP018IV20,U00018
8,9,LEAP01901VI19,LEAP019VI2019,U00019
9,10,LEAP02030IV09,LEAP020IV30,U00020


In [47]:
import pandas as pd
import pickle

print("================================================================================")
print(" 🔄 FASE 4 - UPDATE ID KARYAWAN (VARCHAR -> INT AUTO INCREMENT) 🔄 ")
print("================================================================================")

# 1. LOAD DATA MAPPING DARI FASE 2
# Sesuaikan path-nya jika folder fase_2 ada di direktori yang berbeda
path_mapping = '../fase_2/mapping_id_karyawan_users.pkl' 

try:
    with open(path_mapping, 'rb') as f:
        df_mapping = pickle.load(f)
    print("✓ Berhasil memuat file mapping_id_karyawan_users.pkl")
except FileNotFoundError:
    print(f"❌ File tidak ditemukan di path: {path_mapping}. Tolong sesuaikan path-nya ya, Mut!")

# 2. BUAT KAMUS (DICTIONARY) MAPPING
# Key   : id_user (Varchar lama, misal 'U00012' atau 'LEAP...')
# Value : id_karyawan (Integer auto increment baru)
# --- BAGIAN INI YANG DIUPDATE Sesuai nama kolom barumu ---
dict_mapping_karyawan = dict(zip(df_mapping['id_user'], df_mapping['id_karyawan']))

# 3. EKSEKUSI UPDATE KE TABEL-TABEL TERKAIT

# A. UPDATE IZIN_KARYAWAN
if 'izin_karyawan' in df_new:
    print("\nMemperbarui id_karyawan di tabel izin_karyawan...")
    df_new['izin_karyawan']['id_karyawan'] = df_new['izin_karyawan']['id_karyawan'].map(dict_mapping_karyawan)
    
    # Intip hasil
    display(df_new['izin_karyawan'][['id_izin', 'id_karyawan', 'jenis_izin']])

# B. UPDATE KARYAWAN_RESIGN
if 'karyawan_resign' in df_new:
    print("\nMemperbarui id_karyawan di tabel karyawan_resign...")
    df_new['karyawan_resign']['id_karyawan'] = df_new['karyawan_resign']['id_karyawan'].map(dict_mapping_karyawan)
    
    # Intip hasil
    display(df_new['karyawan_resign'][['id_resign', 'id_karyawan', 'alasan_resign']])

print("\n✅ UPDATE SELESAI! Kolom id_karyawan sekarang sudah berupa Integer.")

 🔄 FASE 4 - UPDATE ID KARYAWAN (VARCHAR -> INT AUTO INCREMENT) 🔄 
❌ File tidak ditemukan di path: ../fase_2/mapping_id_karyawan_users.pkl. Tolong sesuaikan path-nya ya, Mut!

Memperbarui id_karyawan di tabel izin_karyawan...


,id_izin,id_karyawan,jenis_izin
0,P00003,4,Ijin
1,P00004,4,Lembur
2,P00007,5,Lembur
3,P00009,5,Lembur
4,P00010,2,Ijin
...,...,...,...
952,P01002,11,Ijin
953,P01003,11,Ijin
954,P01004,4,Ijin
955,P01005,18,Lembur



Memperbarui id_karyawan di tabel karyawan_resign...


,id_resign,id_karyawan,alasan_resign
0,3,2,menikah dan fokus rumah tangga
1,4,1,Tidak ada keterangan
2,11,3,ikut suami
3,12,4,Tidak ada keterangan
4,14,5,Tidak ada keterangan
5,15,6,Tidak ada keterangan
6,18,7,Tidak ada keterangan
7,20,8,Tidak ada keterangan
8,21,9,Tidak ada keterangan
9,22,10,Tidak ada keterangan



✅ UPDATE SELESAI! Kolom id_karyawan sekarang sudah berupa Integer.


In [48]:
import json
import pickle
with open('fase_4_cimut.pkl', 'wb') as f:
    pickle.dump(df_new, f)

print("✓ Data df_new sudah disimpan ke df_new.pkl")
print("Siap untuk digunakan di insert_handler.ipynb")

✓ Data df_new sudah disimpan ke df_new.pkl
Siap untuk digunakan di insert_handler.ipynb
